# Phase 2：从 BM25 到 Hybrid 检索

## 今天交付什么？

读取 Phase 1 的真实 `chunks.json`，建立一个能解释、能评估的 BM25 检索 baseline；然后用小型数学实验理解 Dense 检索和 RRF 融合为什么可能有价值。今天不把“用了大模型”当作成绩，成绩是：**同一批 Query、同一份 qrels、不同检索策略的可复现实验证据。**

**完成后你要能回答：**

1. BM25 为什么擅长型号、编号和精确词匹配？
2. Dense 检索为什么可能找回词面不同但语义相近的文本？
3. 为什么不能直接把 BM25 分数和 cosine 分数相加？RRF 解决了什么问题？
4. Recall@k 和 MRR@k 分别在衡量什么？

## Evidence Quest 任务卡：Phase 2 总览：赢下搜索对决

**你的身份：** 检索战术总教练  
**案件背景：** 同一个问题会被不同搜索方法排出不同顺序。你要用可解释 baseline 和指标判断谁真的找到了证据。

### 本关专业 Goal

完成 BM25、Dense 机制实验、RRF 融合和 qrels 质量评估。

### 你要交付的作品

**Hybrid 检索对决总览 + 质量成绩单**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：检索战术总教练  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 0. 前置知识与学习顺序

| 知识 | 够用理解 | 立即实验 |
| --- | --- | --- |
| 排序和集合 | 能看 top-k ID，求交集 | 手算 Recall |
| BM25 直觉 | 词频、稀有度、文档长度 | 改 Query 看排名 |
| 向量和 cosine | 方向相似，不只看长度 | 2D 向量手算 |
| RRF | 只使用名次，不比较异构分数 | 看融合贡献 |

先做最小可解释实验，再调用项目模块。这样你知道模块输出为什么长这样，而不是只记住一个 import。

In [1]:
from pathlib import Path
import json
import sys


def find_project_root() -> Path:
    """兼容从项目根目录、notebooks 目录或 JupyterLab 启动目录运行。"""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)

from phase1_doc_parser.main import build_chunks
from phase1_doc_parser.splitter import RecursiveSplitter
from phase2_semantic_search.bm25 import BM25Retriever, tokenize
from phase2_semantic_search.fusion import reciprocal_rank_fusion
from phase2_semantic_search.metrics import evaluate_qrels, recall_at_k, mrr_at_k

chunks_path = ROOT / "data" / "processed" / "chunks.json"
if not chunks_path.exists():
    chunks = build_chunks(ROOT / "phase1_doc_parser" / "examples" / "input", RecursiveSplitter(128, 32))
    chunks_path.parent.mkdir(parents=True, exist_ok=True)
    chunks_path.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    chunks = json.loads(chunks_path.read_text(encoding="utf-8"))
print("读取", len(chunks), "个真实 Chunk")

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship
读取 4 个真实 Chunk


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase2'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase2
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


## 1. 先观察 tokenizer：中文为什么不能照搬英文？

BM25 的输入不是原句，而是一串 token。英文通常按空格或词切分；中文没有天然空格，本项目 baseline 使用“英文单词/数字 + 单个中文字符”的确定性 tokenizer。它的优点是无需外部分词器、不会因为词典版本变化而改变结果；缺点是“检索”可能被拆成多个字符，短语边界和专业词完整性较弱。

这不是哪种 tokenizer 永远正确的问题，而是要用你的 qrels 做选择。

In [3]:
examples = ["BM25 对产品型号更稳", "Dense retrieval 理解语义相近表达", "RRF@10"]
for text in examples:
    print(text, "->", tokenize(text))

BM25 对产品型号更稳 -> ['bm25', '对', '产', '品', '型', '号', '更', '稳']
Dense retrieval 理解语义相近表达 -> ['dense', 'retrieval', '理', '解', '语', '义', '相', '近', '表', '达']
RRF@10 -> ['rrf', '10']


## 2. BM25 原理：把公式翻译成人话

对查询中的每个词，BM25 大致做三件事：

1. **匹配奖励**：这个词在文档出现，分数增加；出现多次会增加，但不会无限增加（`k1` 控制饱和）。
2. **稀有词加权**：只在少数文档出现的词更能区分文档（IDF）。所有文档都有的词贡献较低。
3. **长度归一化**：长文档天然更容易包含词，所以要按平均长度修正（`b` 控制修正程度）。

因此 BM25 的强项很直观：产品型号、错误码、法规编号这种“必须精确出现”的词，通常不应该被语义模型的近义联想替代。

In [4]:
import math

def idf(document_count: int, document_frequency: int) -> float:
    return math.log(1 + (document_count - document_frequency + 0.5) / (document_frequency + 0.5))

for df in (1, 2, 10):
    print(f"N=10, df={df}, IDF={idf(10, df):.3f}")
print("观察：df 越小，词越稀有，区分能力越强。")

N=10, df=1, IDF=1.992
N=10, df=2, IDF=1.482
N=10, df=10, IDF=0.047
观察：df 越小，词越稀有，区分能力越强。


In [5]:
retriever = BM25Retriever(chunks)
queries = {
    "q-001": "Chunk overlap",
    "q-002": "Dense BM25",
}

runs = {}
for query_id, query in queries.items():
    results = retriever.search(query, top_k=5)
    runs[query_id] = [item.doc_id for item in results]
    print(f"\n{query_id}: {query}")
    for rank, item in enumerate(results, start=1):
        print(rank, item.doc_id, round(item.score, 4), item.metadata.get("source"))


q-001: Chunk overlap
1 82ddc7d612e87b02 1.746 D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\quickstart.md
2 3692b05e025373a8 0.7116 D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\quickstart.md

q-002: Dense BM25
1 5a62fff245b307a5 1.6363 D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\retrieval-notes.md
2 83510b23d2680327 1.5321 D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input\retrieval-notes.md


### 如何解释一次 BM25 排名？

不要只说“第一个分数最高”。请按这个顺序解释：

1. 查询被切成了哪些 token？
2. 结果文档命中了哪些 token？
3. 命中的 token 是常见词还是稀有词？
4. 文档长度是否让它受到归一化影响？

这套解释方式比背“BM25 是一种稀疏检索算法”更有用，因为当结果错了，你知道下一步应该检查 tokenizer、数据还是参数。

## 3. Dense 检索：先用二维向量理解，不急着下载模型

Dense 检索把文本映射成向量。相似度常用 cosine：

```text
cos(a,b) = a·b / (||a|| ||b||)
```

它关注向量方向，所以“长度不同但方向相近”的文本仍可能相似。下面用人为定义的二维向量模拟：`猫` 和 `小猫` 方向接近，`汽车型号` 与它们方向不同。这个实验不是模型效果，而是帮助你理解排序机制。

In [6]:
import numpy as np

vectors = {
    "doc-cat": np.array([0.95, 0.20]),
    "doc-kitten": np.array([0.80, 0.35]),
    "doc-car-model": np.array([0.10, 0.99]),
}
query_vector = np.array([1.0, 0.0])

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

dense_scores = {doc_id: cosine(query_vector, vector) for doc_id, vector in vectors.items()}
print(sorted(dense_scores.items(), key=lambda item: item[1], reverse=True))
print("cosine 只比较方向；真实模型负责学习文本到向量的映射。")

[('doc-cat', 0.978549784986749), ('doc-kitten', 0.9161573349021892), ('doc-car-model', 0.1004987059618685)]
cosine 只比较方向；真实模型负责学习文本到向量的映射。


## 4. 为什么要 Hybrid？

BM25 和 Dense 的错误模式不同：

- BM25 可能漏掉“词面不同但意思接近”的表达。
- Dense 可能把语义相近但型号不一样的文档放得太前。

如果两路结果互补，Hybrid 可以先分别召回，再融合排名。这里使用 RRF（Reciprocal Rank Fusion）：

```text
RRF(d) = Σ 1 / (rrf_k + rank(d))
```

它只看名次，不要求 BM25 分数和 cosine 分数处在同一量纲，因此不需要先做不可靠的“分数相加”。

In [7]:
toy_rankings = {
    "bm25": ["exact-model", "semantic-doc", "unrelated"],
    "dense": ["semantic-doc", "exact-model", "another"],
}
fused = reciprocal_rank_fusion(toy_rankings, rrf_k=60)
for item in fused:
    print(item)

print("\n同一个文档出现在两路排名中，会得到两份 rank contribution；这就是互补信号的来源。")

FusedResult(doc_id='exact-model', score=0.03252247488101534, best_rank=1)
FusedResult(doc_id='semantic-doc', score=0.03252247488101534, best_rank=1)
FusedResult(doc_id='another', score=0.015873015873015872, best_rank=3)
FusedResult(doc_id='unrelated', score=0.015873015873015872, best_rank=3)

同一个文档出现在两路排名中，会得到两份 rank contribution；这就是互补信号的来源。


## 5. 用 qrels 评估：相关性必须先被定义

`qrels` 是“Query -> 相关文档 ID”的人工或规则标注。它把“我觉得这个结果不错”变成可计算的证据。

- **Recall@k**：相关文档中有多少进入前 k 名？适合衡量“有没有找全”。
- **MRR@k**：第一个相关结果排第几？`1/rank`，适合衡量“用户第一眼能不能看到”。

当前样例很小，因此 qrels 只作为教学演示。真实项目要扩大 Query、记录标注依据，并单独标记不可回答问题。

In [8]:
first_chunk = chunks[0]["id"]
second_chunk = chunks[1]["id"] if len(chunks) > 1 else first_chunk
qrels = {"q-001": {first_chunk}, "q-002": {second_chunk}}
metrics = evaluate_qrels(runs, qrels, k=5)
print("qrels:", qrels)
print("BM25 metrics:", metrics)
print("手算 q-001:", "Recall=", recall_at_k(runs["q-001"], qrels["q-001"], k=5), "MRR=", mrr_at_k(runs["q-001"], qrels["q-001"], k=5))

qrels: {'q-001': {'3692b05e025373a8'}, 'q-002': {'82ddc7d612e87b02'}}
BM25 metrics: {'recall@5': 0.5, 'mrr@5': 0.25}
手算 q-001: Recall= 1.0 MRR= 0.5


### 一个重要的诚实边界

本 Notebook 当前能真实运行的是 BM25 baseline 和 RRF 接口；二维 Dense 是机制实验，不是 BGE-M3 的效果报告。只有在安装 `requirements/phase2.txt`、下载固定模型、记录 revision 和设备后，才能把真实 Dense 结果写进对比表。

先检查能力是否存在，不因为环境没有模型就偷偷用别的结果冒充：

In [9]:
import importlib.util

optional = {
    "numpy": importlib.util.find_spec("numpy") is not None,
    "faiss": importlib.util.find_spec("faiss") is not None,
    "FlagEmbedding": importlib.util.find_spec("FlagEmbedding") is not None,
}
print(optional)
if not optional["FlagEmbedding"]:
    print("当前环境未安装 BGE-M3；保留可运行 baseline，真实 Dense 放到独立实验。")

{'numpy': True, 'faiss': False, 'FlagEmbedding': False}
当前环境未安装 BGE-M3；保留可运行 baseline，真实 Dense 放到独立实验。


## 6. 小实验：k 改变了什么？

`top_k` 不是越大越好：增大 k 可能提高 Recall，但会增加返回数据、上下文噪声和后续生成成本。下面记录同一批 Query 的结果形状；Phase 3 再把它和延迟放在一起看。

In [10]:
for k in (1, 2, 5):
    per_query = {}
    for query_id, query in queries.items():
        ids = [item.doc_id for item in retriever.search(query, top_k=k)]
        per_query[query_id] = {"ids": ids, "recall": recall_at_k(ids, qrels[query_id], k=k)}
    print("k=", k, per_query)

k= 1 {'q-001': {'ids': ['82ddc7d612e87b02'], 'recall': 0.0}, 'q-002': {'ids': ['5a62fff245b307a5'], 'recall': 0.0}}
k= 2 {'q-001': {'ids': ['82ddc7d612e87b02', '3692b05e025373a8'], 'recall': 1.0}, 'q-002': {'ids': ['5a62fff245b307a5', '83510b23d2680327'], 'recall': 0.0}}
k= 5 {'q-001': {'ids': ['82ddc7d612e87b02', '3692b05e025373a8'], 'recall': 1.0}, 'q-002': {'ids': ['5a62fff245b307a5', '83510b23d2680327'], 'recall': 0.0}}


## Phase 2 阶段闸门

- [ ] 能用自己的话解释 BM25 的匹配、稀有度和长度归一化。
- [ ] 能用二维向量解释 cosine，而不是把 Dense 当作黑盒魔法。
- [ ] 能说明 RRF 为什么融合排名而不是直接融合分数。
- [ ] BM25 结果来自 Phase 1 真实 Chunk，并由 qrels 计算 Recall/MRR。
- [ ] 能明确区分“真实模型结果”和“教学模拟结果”。

**项目交付物：** BM25 baseline、qrels、指标表、一次失败结果的原因分析。下一阶段不追求更复杂模型，而是先证明当前系统的质量和速度到底是多少。

## Boss Challenge：给同一个 Query 展示两种排名，解释为什么最终选择某个检索 baseline。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [11]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [12]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase2_evaluation.json', 'data/processed/phase2_rrf_demo.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase2_evaluation.json', 'data\\processed\\phase2_rrf_demo.json']
